# KisanAI - Market Price Forecasting Agent

KisanAI / Sampoorn Kisan AI Sahayak

## Goal
Forecast the next Modal Price for a commodity at a market, using only *past* prices
(lag features) -- this is what a farmer actually needs: "given the price history so far,
what will it likely be next?"

Important design choice: we do NOT use the same day's Min/Max price to predict the same
day's Modal price. That would be trivial (Modal sits close to the Min/Max average by
definition) and wouldn't be a real forecast. Instead we use lag features built only from
earlier time steps in each commodity-market series.

We also compare against a naive baseline ("tomorrow's price = today's price") -- if the
model can't beat that, it isn't adding real value. This is standard practice for any
forecasting claim and is worth keeping in your report/paper.

In [ ]:
!pip -q install pandas numpy scikit-learn xgboost joblib

## Upload the AGMARKET dataset
Upload `agmarknet_sample.csv`, or `agmarknet_real_data.csv` if you've already run
`fetch_real_agmarknet.py` with your own data.gov.in API key (free signup --
see the comments at the top of that script). Column names from the live API may
differ slightly (often lowercase, e.g. `modal_price`) -- adjust COLUMN_MAP below
if so; the rest of the notebook is written against the standardized names.

In [ ]:
from google.colab import files
uploaded = files.upload()

import glob, pandas as pd
csvs = glob.glob("/content/**/*.csv", recursive=True)
print("CSV files found:", csvs)

CSV_PATH = csvs[0]
df = pd.read_csv(CSV_PATH)
print("Shape:", df.shape)
df.head()

In [ ]:
# Standardize column names if you're using live data.gov.in output instead of the sample
COLUMN_MAP = {
    # "state": "State", "district": "District", "market": "Market",
    # "commodity": "Commodity", "arrival_date": "Date",
    # "min_price": "Min_Price_Rs_per_Quintal", "max_price": "Max_Price_Rs_per_Quintal",
    # "modal_price": "Modal_Price_Rs_per_Quintal",
}
if COLUMN_MAP:
    df = df.rename(columns=COLUMN_MAP)

df["Date"] = pd.to_datetime(df["Date"])
print("Columns:", df.columns.tolist())
print("Date range:", df["Date"].min(), "to", df["Date"].max())
print("Series (Commodity x Market):", df.groupby(["Commodity","Market"]).ngroups)

## Build lag features (leakage-free)

For each Commodity+Market series, sorted by date: lag_1/lag_2/lag_3 = the previous 1/2/3
observed Modal Prices, rolling_mean_3 / rolling_std_3 = stats over the prior 3 observations.
All of these use `.shift()` so the current row's own price is never used to build its own features.

In [ ]:
df = df.sort_values(["Commodity", "Market", "Date"]).reset_index(drop=True)

PRICE_COL = "Modal_Price_Rs_per_Quintal"
grp = df.groupby(["Commodity", "Market"])[PRICE_COL]

df["lag_1"] = grp.shift(1)
df["lag_2"] = grp.shift(2)
df["lag_3"] = grp.shift(3)
group_key = ["Commodity", "Market"]
df["rolling_mean_3"] = df.groupby(group_key)[PRICE_COL].transform(lambda s: s.shift(1).rolling(3).mean())
df["rolling_std_3"] = df.groupby(group_key)[PRICE_COL].transform(lambda s: s.shift(1).rolling(3).std())

df["month"] = df["Date"].dt.month
df["day_of_year"] = df["Date"].dt.dayofyear

# Rows at the start of each series don't have 3 prior observations yet -- drop them
model_df = df.dropna(subset=["lag_1", "lag_2", "lag_3", "rolling_mean_3", "rolling_std_3"]).reset_index(drop=True)
print("Rows before dropping short series starts:", len(df))
print("Rows usable for training:", len(model_df))

## Encode categoricals and do a time-based train/test split

A random split would leak future prices into training (a row from March could train the
model that then gets tested on January). We split by date instead: the most recent ~20%
of dates become the test set, mimicking real deployment (predict the future from the past).

In [ ]:
from sklearn.preprocessing import OneHotEncoder

CAT_COLS = ["Commodity", "Market", "State"]
NUM_COLS = ["lag_1", "lag_2", "lag_3", "rolling_mean_3", "rolling_std_3", "month", "day_of_year"]

encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
cat_encoded = encoder.fit_transform(model_df[CAT_COLS])
cat_df = pd.DataFrame(cat_encoded, columns=encoder.get_feature_names_out(CAT_COLS))

X = pd.concat([model_df[NUM_COLS].reset_index(drop=True), cat_df], axis=1)
y = model_df[PRICE_COL].reset_index(drop=True)

cutoff_date = model_df["Date"].quantile(0.8, interpolation="nearest")
train_mask = (model_df["Date"] <= cutoff_date).values

X_train, X_test = X[train_mask], X[~train_mask]
y_train, y_test = y[train_mask], y[~train_mask]
dates_test = model_df.loc[~train_mask, "Date"]

print("Cutoff date:", cutoff_date)
print("Train rows:", len(X_train), "| Test rows:", len(X_test))

## Train the model, and the naive baseline

In [ ]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

model = XGBRegressor(
    n_estimators=300, max_depth=5, learning_rate=0.05, random_state=42
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# Naive baseline: predict "no change" -- next price = lag_1 (the most recent known price)
baseline_pred = model_df.loc[~train_mask, "lag_1"].values

def report(name, y_true, y_hat):
    mae = mean_absolute_error(y_true, y_hat)
    rmse = np.sqrt(mean_squared_error(y_true, y_hat))
    r2 = r2_score(y_true, y_hat)
    print(f"{name:20s}  MAE: {mae:8.2f}   RMSE: {rmse:8.2f}   R2: {r2:.3f}")

print("--- Model vs naive baseline on held-out future dates ---")
report("XGBoost model", y_test, y_pred)
report("Naive (no-change)", y_test, baseline_pred)

If the model's MAE/RMSE isn't meaningfully better than the naive baseline, that's an honest
result worth reporting too -- short-term mandi prices are often close to a random walk, and
beating "no change" by even a modest margin, consistently, is a real result.

In [ ]:
# Feature importance -- which factors actually drive the forecast
importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
print(importances.head(15))

## Save model + encoder

In [ ]:
import joblib, os

os.makedirs("/content/kisanai_models", exist_ok=True)

artifact = {
    "model": model,
    "encoder": encoder,
    "cat_cols": CAT_COLS,
    "num_cols": NUM_COLS,
    "price_col": PRICE_COL,
}

MODEL_PATH = "/content/kisanai_models/market_price_model.pkl"
joblib.dump(artifact, MODEL_PATH)
print("Saved:", MODEL_PATH)

## MarketAgent class -- ready for the Orchestrator

Takes a commodity/market's recent price history (at least 3 past observations) and
forecasts the next Modal Price. Replaces the placeholder `MarketAgent` stub in
`05_KisanAI_Orchestrator.ipynb`.

In [ ]:
class MarketAgent:
    def __init__(self, artifact):
        self.model = artifact["model"]
        self.encoder = artifact["encoder"]
        self.cat_cols = artifact["cat_cols"]
        self.num_cols = artifact["num_cols"]

    def forecast(self, commodity, market, state, recent_prices, target_date):
        """
        recent_prices: list of the last 3+ known Modal Prices, most recent LAST.
        target_date: a pandas.Timestamp / datetime for the date being forecast.
        """
        if len(recent_prices) < 3:
            return {"error": "Need at least 3 past price observations to forecast."}

        last3 = recent_prices[-3:]
        row = pd.DataFrame([{
            "Commodity": commodity, "Market": market, "State": state,
            "lag_1": last3[-1], "lag_2": last3[-2], "lag_3": last3[-3],
            "rolling_mean_3": np.mean(last3), "rolling_std_3": np.std(last3),
            "month": target_date.month, "day_of_year": target_date.dayofyear,
        }])

        cat_part = pd.DataFrame(
            self.encoder.transform(row[self.cat_cols]),
            columns=self.encoder.get_feature_names_out(self.cat_cols)
        )
        x = pd.concat([row[self.num_cols].reset_index(drop=True), cat_part], axis=1)
        pred = float(self.model.predict(x)[0])

        trend = "rising" if pred > last3[-1] else "falling" if pred < last3[-1] else "stable"
        return {"forecast_modal_price": round(pred, 2), "trend": trend, "last_known_price": last3[-1]}


# Quick manual test using a real series from the data
agent = MarketAgent(artifact)
sample_series = model_df[(model_df["Commodity"] == model_df["Commodity"].iloc[0]) &
                          (model_df["Market"] == model_df["Market"].iloc[0])].sort_values("Date")
last_row = sample_series.iloc[-1]
recent = [last_row["lag_3"], last_row["lag_2"], last_row["lag_1"]]

result = agent.forecast(
    commodity=last_row["Commodity"], market=last_row["Market"], state=last_row["State"],
    recent_prices=recent, target_date=last_row["Date"]
)
print("Forecast:", result)
print("Actual price on that date was:", last_row[PRICE_COL])

## Download model for the local KisanAI project

In [ ]:
from google.colab import files
files.download(MODEL_PATH)

## Next steps
1. Register a free data.gov.in API key and run `fetch_real_agmarknet.py` to get a much
   larger, continuously-updating dataset instead of the fixed sample.
2. Add `MarketAgent` to `05_KisanAI_Orchestrator.ipynb` alongside `CropAgent`, replacing
   the placeholder stub.
3. Report both the model's MAE/RMSE *and* the naive baseline's in your review/paper --
   showing you beat a proper baseline is stronger evidence than accuracy alone.